[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_04_california_housing/task_4_california_housing_mlp.ipynb)

# Week 4 · MLP for regression: California housing

Predict the median house value of a Californian census block group from 8 numeric features. Metric: RMSE (root mean squared error) in units of 100 000 USD, lower is better.

Working in Colab? Replace `fiit-ba` in the badge URL with your GitHub username to open the copy in your fork, and run the setup cell below.

In [ ]:
# Colab setup (does nothing when you run locally)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb

# --- Configuration ---
SEED = 42
SUBSET = None        # e.g. 256 while debugging: train on only that many rows (see labs/README.md); None = everything
DATA_DIR = Path(os.environ.get("ZNEUS_DATA_DIR", "data"))

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## Data

Download the competition files from the *Data* tab of the Kaggle competition (link given at the lab) into `data/kaggle/` next to this notebook (or into `$ZNEUS_DATA_DIR/kaggle/`). With the Kaggle CLI: `uv run kaggle competitions download -c <competition-slug> -p data/kaggle` and unzip there.

| file | content |
|---|---|
| `train.csv` | 16 512 rows: `id`, the 8 features, the target `MedHouseVal` |
| `test.csv` | 4 128 rows: `id` and the 8 features, no target |
| `sample_submission.csv` | the required format: `id,MedHouseVal` |

| column | meaning |
|---|---|
| `MedInc` | median income of the block group, in tens of thousands of USD |
| `HouseAge` | median age of the houses |
| `AveRooms`, `AveBedrms` | average number of rooms / bedrooms per household |
| `Population` | number of people in the block group |
| `AveOccup` | average household size |
| `Latitude`, `Longitude` | location |
| `MedHouseVal` | **target**: median house value in units of 100 000 USD |

The cell below loads the files into `train_df` / `test_df` and the arrays `X_all`, `y_all` (all labelled rows) and `X_test` (the rows to predict, in the order of `test_df["id"]`). Without the Kaggle files it falls back to scikit-learn's copy of the dataset with a local 80/20 split, so the notebook runs anywhere; that fallback has different ids, so its `submission.csv` will not score on Kaggle.

In [ ]:
FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
TARGET = "MedHouseVal"
KAGGLE_DIR = DATA_DIR / "kaggle"

if (KAGGLE_DIR / "train.csv").exists() and (KAGGLE_DIR / "test.csv").exists():
    train_df = pd.read_csv(KAGGLE_DIR / "train.csv")
    test_df = pd.read_csv(KAGGLE_DIR / "test.csv")
    y_test_local = None
    print("Kaggle files found in", KAGGLE_DIR)
else:
    from sklearn.datasets import fetch_california_housing

    frame = fetch_california_housing(as_frame=True, data_home=DATA_DIR / "sklearn").frame
    frame.insert(0, "id", np.arange(len(frame)))
    perm = np.random.default_rng(SEED).permutation(len(frame))
    n_test = len(frame) // 5
    test_rows = frame.iloc[perm[:n_test]]
    train_rows = frame.iloc[perm[n_test:]]
    train_df = train_rows.reset_index(drop=True)
    test_df = test_rows.drop(columns=[TARGET]).reset_index(drop=True)
    y_test_local = test_rows[TARGET].to_numpy(dtype=np.float32)
    print(f"No Kaggle files in {KAGGLE_DIR} -> scikit-learn copy with a local 80/20 split (its ids are NOT the Kaggle ids)")

if SUBSET is not None:
    train_df = train_df.head(SUBSET)

X_all = train_df[FEATURES].to_numpy(dtype=np.float32)
y_all = train_df[TARGET].to_numpy(dtype=np.float32)
X_test = test_df[FEATURES].to_numpy(dtype=np.float32)
print(f"X_all {X_all.shape}   y_all {y_all.shape}   X_test {X_test.shape}")

## Your task

Train the best MLP you can for this regression task, log every run to Weights & Biases (project `zneus-2026`, run names `week04-...`) and submit your test predictions to Kaggle. Your code goes into the cell below; it must end with `test_pred` (a numpy array, a list or a torch tensor), one prediction per row of `X_test`, in the same order. The last cell writes `submission.csv`.

`wandb login` once in a terminal (API key from https://wandb.ai/authorize); without an account set `WANDB_MODE=offline` and `wandb sync` the runs later. Set the project `zneus-2026` to **Public** before you hand in.

In [ ]:
# TODO: your solution. When this cell has run, `test_pred` must hold one prediction per row of X_test.
test_pred = ...

## Kaggle submission

`submission.csv` needs exactly the columns `id,MedHouseVal`, one row per row of `test.csv`, no index column. Upload it on the Kaggle competition page (*Submit Predictions*) or with `uv run kaggle competitions submit -c <competition-slug> -f submission.csv -m "week 4"`.

In [ ]:
assert test_pred is not ..., "fill in the solution cell above: test_pred is still `...`"
if torch.is_tensor(test_pred):
    test_pred = test_pred.detach().cpu().numpy()
test_pred = np.asarray(test_pred, dtype=np.float32).reshape(-1)
assert len(test_pred) == len(test_df), f"test_pred has {len(test_pred)} values, expected one per test row ({len(test_df)})"
assert np.isfinite(test_pred).all(), "test_pred contains NaN or inf"

submission = pd.DataFrame({"id": test_df["id"].to_numpy(), TARGET: test_pred})
submission.to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(submission)} rows, columns {list(submission.columns)}")
if y_test_local is not None:
    local_rmse = np.sqrt(np.mean((test_pred - y_test_local) ** 2))
    print(f"RMSE on the local held-out split: {local_rmse:.4f}")
submission.head()

## Before you hand in

- [ ] the notebook runs top to bottom and is committed and pushed to your fork (`data/`, `wandb/` and `submission.csv` are git-ignored, leave them out),
- [ ] `submission.csv` is on the Kaggle competition leaderboard under your AIS login,
- [ ] your W&B project `zneus-2026` is public and contains your runs; hand in the link.